# Advanced RAG Demo

Build an advanced Retrieval-Augmented Generation system over Hugging Face documentation.

Improvements over simple RAG:

- Markdown-aware, token-aware chunking
- Chunk-length diagnostics and duplicate removal
- Normalized cosine retrieval with FAISS
- 2D embedding-space visualization
- Model-specific chat templating
- Retrieve-many, rerank, keep-few retrieval
- Source-aware answers and a small evaluation loop

## 1. Use a Colab GPU

Select **Runtime > Change runtime type > T4 GPU**. The reader model uses 4-bit quantization.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu \
  datasets langchain langchain-community langchain-text-splitters \
  langchain-huggingface ragatouille pacmap plotly

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

pd.set_option("display.max_colwidth", None)

## 2. Load the knowledge base

The source recipe processes the complete dataset. For a predictable classroom runtime, this notebook uses 300 source documents. Set `MAX_SOURCE_DOCS = None` to process the entire corpus.

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document

MAX_SOURCE_DOCS = 300

dataset = load_dataset("m-ric/huggingface_doc", split="train")
if MAX_SOURCE_DOCS is not None:
    dataset = dataset.select(range(min(MAX_SOURCE_DOCS, len(dataset))))

raw_knowledge_base = [
    Document(page_content=row["text"], metadata={"source": row["source"]})
    for row in tqdm(dataset)
]

print("Source documents:", len(raw_knowledge_base))
print("Example source:", raw_knowledge_base[0].metadata["source"])
print(raw_knowledge_base[0].page_content[:700])

## 3. First attempt: Markdown-aware character chunking

This preserves headings, code fences, thematic breaks, paragraphs, lines, and words where possible.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

character_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    add_start_index=True,
    strip_whitespace=True,
    separators=MARKDOWN_SEPARATORS,
)

character_chunks = []
for doc in raw_knowledge_base:
    character_chunks.extend(character_splitter.split_documents([doc]))

print("Character-based chunks:", len(character_chunks))

Character limits do not guarantee that chunks fit the embedding model's token limit. Measure with the embedding tokenizer.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

EMBEDDING_MODEL_NAME = "thenlper/gte-small"
embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
embedding_backbone = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model maximum sequence length:", embedding_backbone.max_seq_length)

character_token_lengths = [
    len(embedding_tokenizer.encode(doc.page_content))
    for doc in tqdm(character_chunks)
]

pd.Series(character_token_lengths).hist(bins=30)
plt.axvline(embedding_backbone.max_seq_length, color="red", linestyle="--")
plt.title("Character chunks measured in embedding-model tokens")
plt.xlabel("Token count")
plt.show()

print("Chunks above model limit:", sum(n > embedding_backbone.max_seq_length for n in character_token_lengths))

## 4. Improved token-aware chunking and deduplication

Count chunk size with the actual embedding tokenizer, overlap by 10%, preserve source/start metadata, and remove exact duplicate chunk text.

In [ ]:
def split_documents(chunk_size, knowledge_base, tokenizer_name=EMBEDDING_MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer,
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size / 10),
        add_start_index=True,
        strip_whitespace=True,
        separators=MARKDOWN_SEPARATORS,
    )

    chunks = []
    for doc in knowledge_base:
        chunks.extend(splitter.split_documents([doc]))

    unique_chunks = []
    seen_text = set()
    for doc in chunks:
        if doc.page_content not in seen_text:
            seen_text.add(doc.page_content)
            unique_chunks.append(doc)

    return unique_chunks

docs_processed = split_documents(512, raw_knowledge_base)
token_lengths = [
    len(embedding_tokenizer.encode(doc.page_content))
    for doc in tqdm(docs_processed)
]

pd.Series(token_lengths).hist(bins=30)
plt.axvline(512, color="red", linestyle="--")
plt.title("Token-aware chunk distribution")
plt.xlabel("Token count")
plt.show()

print("Unique chunks:", len(docs_processed))
print("Maximum measured length:", max(token_lengths))

## 5. Build normalized cosine embeddings and FAISS

Normalize vectors and explicitly configure cosine distance.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

knowledge_vector_database = FAISS.from_documents(
    docs_processed,
    embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

print("Indexed chunks:", knowledge_vector_database.index.ntotal)

## 6. Visualize a sample of embedding space with PaCMAP

Projection is explanatory, not part of production retrieval. To keep Colab responsive, project at most 500 chunk vectors plus the query.

In [ ]:
import pacmap
import plotly.express as px

user_query = "How to create a pipeline object?"
query_vector = embedding_model.embed_query(user_query)

projection_count = min(500, len(docs_processed))
projected_docs = docs_processed[:projection_count]
vectors = [
    knowledge_vector_database.index.reconstruct(i).tolist()
    for i in range(projection_count)
] + [query_vector]

projector = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=None,
    MN_ratio=0.5,
    FP_ratio=2.0,
    random_state=1,
)
points = projector.fit_transform(np.array(vectors), init="pca")

rows = [
    {
        "x": points[i, 0],
        "y": points[i, 1],
        "source": projected_docs[i].metadata["source"].split("/")[1],
        "extract": projected_docs[i].page_content[:100] + "...",
        "kind": "Chunk",
        "size": 5,
    }
    for i in range(projection_count)
]
rows.append({
    "x": points[-1, 0],
    "y": points[-1, 1],
    "source": "User query",
    "extract": user_query,
    "kind": "Query",
    "size": 20,
})

projection_df = pd.DataFrame(rows)
fig = px.scatter(
    projection_df,
    x="x",
    y="y",
    color="source",
    symbol="kind",
    size="size",
    hover_data="extract",
    title="PaCMAP projection of chunk and query embeddings",
    width=1000,
    height=650,
)
fig.show()

## 7. Inspect baseline vector retrieval

In [ ]:
retrieved_docs = knowledge_vector_database.similarity_search(
    query=user_query,
    k=5,
)

for index, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- DOCUMENT {index} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:900])

## 8. Load the 4-bit reader model

The source recipe uses Zephyr-7B. `float16` is used here for Colab T4 compatibility.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

READER_MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

reader_tokenizer = AutoTokenizer.from_pretrained(READER_MODEL_NAME)
reader_model = AutoModelForCausalLM.from_pretrained(
    READER_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
reader_model.eval()

reader_llm = pipeline(
    task="text-generation",
    model=reader_model,
    tokenizer=reader_tokenizer,
    do_sample=True,
    temperature=0.2,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=350,
)

print(reader_llm("What is 4+4? Answer briefly:")[0]["generated_text"])

## 9. Build the prompt with the model's chat template

In [ ]:
prompt_in_chat_format = [
    {
        "role": "system",
        "content": (
            "Use only the information in the context to answer the question. "
            "Be concise and relevant. Cite source document numbers when useful. "
            "If the answer cannot be deduced from the context, say that you do not know."
        ),
    },
    {
        "role": "user",
        "content": "Context:\n{context}\n---\nQuestion: {question}",
    },
]

RAG_PROMPT_TEMPLATE = reader_tokenizer.apply_chat_template(
    prompt_in_chat_format,
    tokenize=False,
    add_generation_prompt=True,
)

print(RAG_PROMPT_TEMPLATE)

## 10. Load the ColBERT reranker

The bi-encoder retrieves many candidates cheaply. ColBERT models finer query-token/document-token interactions and reranks those candidates before generation.

In [ ]:
from ragatouille import RAGPretrainedModel

reranker = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
print("Reranker loaded.")

## 11. Assemble retrieval, optional reranking, prompting, and generation

In [ ]:
def answer_with_rag(
    question,
    llm,
    knowledge_index,
    reranker=None,
    num_retrieved_docs=20,
    num_docs_final=5,
):
    print("=> Retrieving documents...")
    candidates = knowledge_index.similarity_search(
        query=question,
        k=num_retrieved_docs,
    )

    candidate_records = [
        {"content": doc.page_content, "metadata": doc.metadata}
        for doc in candidates
    ]

    if reranker is not None:
        print("=> Reranking documents...")
        reranked = reranker.rerank(
            question,
            [item["content"] for item in candidate_records],
            k=num_docs_final,
        )
        final_records = []
        for result in reranked:
            result_index = result.get("result_index")
            if result_index is not None:
                original = candidate_records[result_index]
            else:
                original = next(
                    item for item in candidate_records
                    if item["content"] == result["content"]
                )
            final_records.append({
                "content": result["content"],
                "metadata": original["metadata"],
                "rerank_score": result["score"],
            })
    else:
        final_records = candidate_records[:num_docs_final]

    context = "\n\n".join(
        f"Document {index}:\n{item['content']}"
        for index, item in enumerate(final_records, start=1)
    )
    final_prompt = RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
    )

    print("=> Generating answer...")
    answer = llm(final_prompt)[0]["generated_text"]
    return answer, final_records


## 12. Compare vector-only retrieval with reranked retrieval

In [ ]:
question = "How to create a pipeline object?"

answer_without_reranking, docs_without_reranking = answer_with_rag(
    question,
    reader_llm,
    knowledge_vector_database,
    reranker=None,
    num_retrieved_docs=5,
    num_docs_final=5,
)

print("\nANSWER WITHOUT RERANKING\n", answer_without_reranking)

In [ ]:
answer_with_reranking, docs_with_reranking = answer_with_rag(
    question,
    reader_llm,
    knowledge_vector_database,
    reranker=reranker,
    num_retrieved_docs=20,
    num_docs_final=5,
)

print("\nANSWER WITH RERANKING\n", answer_with_reranking)
print("\nFINAL SOURCES")
for index, item in enumerate(docs_with_reranking, start=1):
    print(index, item["metadata"].get("source"), item.get("rerank_score"))

## 13. Small evaluation loop

Advanced RAG must be measured. This lightweight table checks retrieval and answer evidence manually. A real project should use a larger labeled evaluation set and automated metrics.

In [ ]:
evaluation_questions = [
    "How to create a pipeline object?",
    "What is the purpose of an attention mask?",
    "How can I load a dataset from the Hugging Face Hub?",
]

evaluation_rows = []
for evaluation_question in evaluation_questions:
    answer, source_records = answer_with_rag(
        evaluation_question,
        reader_llm,
        knowledge_vector_database,
        reranker=reranker,
        num_retrieved_docs=20,
        num_docs_final=5,
    )
    evaluation_rows.append({
        "question": evaluation_question,
        "answer": answer,
        "sources": [item["metadata"].get("source") for item in source_records],
        "retrieval_relevant": "REVIEW",
        "answer_supported": "REVIEW",
    })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df

For each row, read the sources and replace `REVIEW` with `YES` or `NO`. Tune one variable at a time: chunking, embedding model, retrieval depth, reranking, prompt, or reader model.

## Key takeaway

Advanced RAG is not one special model. It is disciplined optimization of the complete system: corpus, token-aware chunks, embeddings, similarity metric, candidate retrieval, reranking, context construction, model-specific prompting, reader generation, source inspection, and evaluation.